In [1]:
%%capture
%pip install pycox lifelines scikit-survival optuna tqdm

In [2]:
import gc
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torchtuples as tt
import xgboost as xgb
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from pycox.models import CoxPH
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from sksurv.datasets import load_breast_cancer
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from tqdm.auto import tqdm
from pycox.models import DeepHitSingle
from pycox.preprocessing.label_transforms import LabTransDiscreteTime
from sksurv.datasets import load_whas500
from lifelines.datasets import load_rossi
from sksurv.metrics import integrated_brier_score
from pycox.models import CoxTime
from pycox.preprocessing.label_transforms import LabTransCoxTime
from pycox.evaluation import EvalSurv
import google
from pycox.models.cox_time import MLPVanillaCoxTime
from sklearn.model_selection import cross_val_score

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

df_colorectal = pd.read_csv('/content/drive/MyDrive/colorectal_data_cleaned.csv')

Mounted at /content/drive


In [4]:
df_colorectal.shape

(407892, 33)

In [5]:
from sklearn.preprocessing import MinMaxScaler

# scale continuous features to [0, 1]
continuous_features = ["Age recode with single ages and 90+", "Regional nodes positive (1988+)",
                    "Regional nodes examined (1988+)", "Tumor_Size"]
scaler = MinMaxScaler()
df_colorectal[continuous_features] = scaler.fit_transform(df_colorectal[continuous_features])

In [6]:
print(df_colorectal[continuous_features].describe().loc[['min', 'max']])

     Age recode with single ages and 90+  Regional nodes positive (1988+)  \
min                                  0.0                              0.0   
max                                  1.0                              1.0   

     Regional nodes examined (1988+)  Tumor_Size  
min                              0.0         0.0  
max                              1.0         1.0  


In [7]:
df_colorectal.head()

,Age recode with single ages and 90+,Regional nodes positive (1988+),Regional nodes examined (1988+),Tumor_Size,T,E,Grade_Ordinal,Stage_Ordinal,Is_Married,Sex_Male,...,Anatomy_Group_Right_Colon,Anatomy_Group_Unknown,Surgery_Group_No_Surgery,Surgery_Group_Partial_Colectomy,Surgery_Group_Radical_or_NOS,Surgery_Group_Total_Colectomy,Surgery_Group_Unknown,CEA_Result_20.0,CEA_Result_30.0,CEA_Result_Unknown
0,0.433333,0.010309,0.084211,0.035389,179.0,0,2,4,0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.911111,0.000000,0.000000,0.040445,0.0,1,3,3,0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.744444,0.000000,0.000000,0.050556,102.0,0,2,3,0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.711111,0.000000,0.073684,0.040445,218.0,0,2,1,0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0.777778,0.010309,0.105263,0.015167,134.0,0,2,2,0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [8]:
# extract features and target variables
X_colorectal = df_colorectal.drop(columns=["T", "E"]).values.astype("float32")
T_colorectal = df_colorectal["T"].values.astype("float32")
E_colorectal = df_colorectal["E"].values.astype("float32")

In [9]:
# train/test split
X_train, X_test, T_train, T_test, E_train, E_test = train_test_split(
    X_colorectal, T_colorectal, E_colorectal, test_size=0.2, random_state=42, stratify=E_colorectal
)

In [10]:
print(f"Train/Test split: {X_train.shape[0]}, {X_test.shape[0]}")
print("Total censoring")
print((E_colorectal==1).sum(), E_colorectal.shape[0] - (E_colorectal==1).sum(), (E_colorectal==0).sum())
print((E_colorectal==0).sum()/E_colorectal.shape[0])

print("Train censoring")
print((E_train==1).sum(), E_train.shape[0] - (E_train==1).sum(), (E_train==0).sum())
print((E_train==0).sum()/E_train.shape[0])

print("Test censoring")
print((E_test==1).sum(), E_test.shape[0] - (E_test==1).sum(), (E_test==0).sum())
print((E_test==0).sum()/E_test.shape[0])


Train/Test split: 326313, 81579
Total censoring
250386 157506 157506
0.38614633285281397
Train censoring
200308 126005 126005
0.3861476557783478
Test censoring
50078 31501 31501
0.38614104119932824


In [11]:
# subsampling for hyperparameter tuning
X_tune, _, T_tune, _, E_tune, _ = train_test_split(
    X_train, T_train, E_train,
    train_size=0.1, # use 10% of training data for tuning (~80000 samples)
    random_state=1,
    stratify=E_train
)

# 80/20 split for tuning
X_tr, X_val, T_tr, T_val, E_tr, E_val = train_test_split(
    X_tune, T_tune, E_tune,
    test_size=0.2,
    random_state=1,
    stratify=E_tune
)

In [12]:
# cox proportional hazards
def cph(X_train, T_train, E_train, X_test, times):
    df_train = pd.DataFrame(X_train)
    df_train["time"] = T_train
    df_train["event"] = E_train

    cph = CoxPHFitter(penalizer=0.0)  # no penalisation

    try:
        cph.fit(df_train, duration_col="time", event_col="event")

        # predictions flattened to a 1D array
        preds = cph.predict_partial_hazard(pd.DataFrame(X_test)).values.flatten()

        # return 2d  array for IBS
        surv = cph.predict_survival_function(pd.DataFrame(X_test), times=times).values.T

        return preds, surv

    except Exception as e:
        print(f"CPH failed to converge: {e}")
        return None

In [13]:
def cox_lasso(X_train, T_train, E_train, X_test, times=None):
    try:
        # scikit-survival requires a structured array for the target variable
        y_train = np.array([(bool(e), t) for e, t in zip(E_train, T_train)],
                           dtype=[('event', 'bool'), ('time', 'float32')])

        # Initialise Lasso (l1_ratio=1.0 is pure Lasso / L1 penalty)
        lasso_model = CoxnetSurvivalAnalysis(l1_ratio=1.0, fit_baseline_model=True)

        # fit model
        lasso_model.fit(X_train, y_train)

        # predict and flatten to ensure a clean 1D array for the bootstrap loop
        risk_scores = lasso_model.predict(X_test).flatten()

        # 2d array of survival functions for IBS
        surv_funcs = lasso_model.predict_survival_function(X_test)
        surv = np.array([fn(times) for fn in surv_funcs])

        return risk_scores, surv

    except Exception as e:
        print(f"Cox Lasso failed to converge: {e}")
        return None

In [14]:
# random survival forest
def rsf(X_train, T_train, E_train, X_test, params, times=None):
    try:
        y_train = np.empty(len(E_train), dtype=[("e", bool), ("t", float)])
        y_train["e"] = E_train.astype(bool)
        y_train["t"] = T_train
        # Apply parameters dynamically, falling back to original defaults if keys are missing
        rsf_model = RandomSurvivalForest(
            n_estimators=params.get("n_estimators", 100),
            min_samples_split=params.get("min_samples_split", 2),
            min_samples_leaf=params.get("min_samples_leaf", 15),
            max_features=params.get("max_features", "sqrt"),
            n_jobs=1,
            verbose=2,
            random_state=1
        )
        rsf_model.fit(X_train, y_train)

        # risk for c-index
        risk = rsf_model.predict(X_test)

        # survival functions for IBS
        surv_funcs = rsf_model.predict_survival_function(X_test)

        if times is not None:
            surv = np.array([fn(times) for fn in surv_funcs]).astype(np.float32)
        else:
            surv = None

        return rsf_model, risk, surv

    except Exception as e:
        print(f"RSF failed: {e}")
        return None, None, None

In [24]:
def optimise_rsf(X_tr, T_tr, E_tr, X_val, T_val, E_val, n_trials=20):
    # Format the split target variables into the structured array required by sksurv
    y_tr = np.array(
        [(bool(e), float(t)) for e, t in zip(E_tr, T_tr)],
        dtype=[("e", bool), ("t", float)]
    )
    y_val = np.array(
        [(bool(e), float(t)) for e, t in zip(E_val, T_val)],
        dtype=[("e", bool), ("t", float)]
    )

    def objective(trial):
        # Define the search space
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 150, step=50),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 3, 20),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None])
        }

        # Initialise and fit on the 80% sub-training set
        rsf = RandomSurvivalForest(**params, n_jobs=-1, random_state=1)
        rsf.fit(X_tr, y_tr)

        # Evaluate c-index on the 20% validation set
        c_index = rsf.score(X_val, y_val)

        # Memory cleanup between trials
        del rsf
        gc.collect()

        return c_index

    # Run Optuna Study
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize")

    study.optimize(objective, n_trials=n_trials, n_jobs=-1, show_progress_bar=True)

    # Return only the parameters
    return study.best_params

In [25]:
def deepsurv(X_train, T_train, E_train, X_test, params, times):
    try:
        # Ensure arrays are float32 for PyTorch compatibility
        X_train = X_train.astype("float32")
        T_train = T_train.astype("float32")
        E_train = E_train.astype("float32")
        X_test = X_test.astype("float32")

        # Split strictly within the training set for Early Stopping
        X_tr, X_val, T_tr, T_val, E_tr, E_val = train_test_split(
            X_train, T_train, E_train, test_size=0.2, random_state=1, stratify=E_train
        )

        y_tr = (T_tr, E_tr)
        y_val = (T_val, E_val)

        # Network features
        in_features = X_tr.shape[1]
        out_features = 1
        num_nodes = [params["nodes"]] * params["layers"]
        dropout = params["dropout"]
        batch_norm = params.get("batch_norm", True)

        # Activation function
        activation = nn.SELU if params.get("activation", "ReLU") == "SELU" else nn.ReLU

        net = tt.practical.MLPVanilla(
            in_features, num_nodes, out_features, batch_norm,
            dropout, activation=activation, output_bias=False
        )

        if params.get("optimiser", "adam") == "adam":
            optim = tt.optim.Adam(lr=params["lr"], weight_decay=params.get("l2", 0.0))
        elif params.get("optimiser") == "sgd":
            optim = tt.optim.SGD(lr=params["lr"], momentum=params.get("momentum", 0.9),
                                 weight_decay=params.get("l2", 0.0), nesterov=True)
        else:
            raise ValueError("Unknown optimiser")

        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = CoxPH(net, optim, device=device)

        callbacks = [tt.callbacks.EarlyStopping(patience=15)]
        batch_size = params.get("batch_size", 64)


        # Fit model
        model.fit(
            X_tr, y_tr,
            batch_size=batch_size,
            epochs=500,
            callbacks=callbacks,
            verbose=False,
            val_data=(X_val, y_val)
        )

        # Return the flat array of log-hazard (risk) scores
        risk = model.predict_net(X_test).flatten()

        # survival probabilities
        _ = model.compute_baseline_hazards(X_tr, y_tr)  # calculate baseline hazards
        surv_df = model.predict_surv_df(X_test)
        surv = surv_df.reindex(times, method='ffill').bfill().values.T

        return model, risk, surv

    except Exception as e:
        print(f"DeepSurv failed: {e}")
        return None, None, None

In [26]:
def optimise_deepsurv(X_tr, T_tr, E_tr, X_val, T_val, E_val, n_trials=20):
    def objective(trial):
        # parameters to optimise
        n_nodes = trial.suggest_int("nodes", 16, 256, step=16)
        n_layers = trial.suggest_int("layers", 1, 4)
        dropout = trial.suggest_float("dropout", 0.1, 0.6)
        lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
        l2 = trial.suggest_float("l2", 1e-4, 1e-1, log=True)
        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128, 256])
        activation_name = trial.suggest_categorical("activation", ["ReLU", "SELU"])
        batch_norm = trial.suggest_categorical("batch_norm", [True, False])

        in_features = X_tr.shape[1]
        out_features = 1
        num_nodes = [n_nodes] * n_layers
        activation = nn.SELU if activation_name == "SELU" else nn.ReLU

        # deepsurv
        net = tt.practical.MLPVanilla(
            in_features, num_nodes, out_features,
            batch_norm=batch_norm, dropout=dropout,
            activation=activation, output_bias=False
        )

        device = "cuda" if torch.cuda.is_available() else "cpu"
        optim = tt.optim.Adam(lr=lr, weight_decay=l2)
        model = CoxPH(net, optim, device=device)

        callbacks = [tt.callbacks.EarlyStopping(patience=15)]

        model.fit(
            X_tr, (T_tr, E_tr),
            batch_size=batch_size, epochs=500,
            callbacks=callbacks, verbose=False,
            val_data=(X_val, (T_val, E_val))
        )

        # PyCox native evaluation
        _ = model.compute_baseline_hazards()
        surv = model.predict_surv_df(X_val)
        ev = EvalSurv(surv, T_val, E_val, censor_surv="km")
        c_index = ev.concordance_td()

        # memory cleanup
        del model, net, optim
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        return c_index

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True)

    return study.best_params

In [27]:
def coxtime(X_train, T_train, E_train, X_test, params, times):
    try:
        labtrans = LabTransCoxTime()
        y_train_cox = labtrans.fit_transform(T_train, E_train)

        X_tr, X_val, y_tr_time, y_val_time, y_tr_event, y_val_event = train_test_split(
            X_train, y_train_cox[0], y_train_cox[1],
            test_size=0.2, random_state=1, stratify=y_train_cox[1]
        )

        y_tr = (y_tr_time, y_tr_event)
        y_val = (y_val_time, y_val_event)

        # Use native MLPVanillaCoxTime for Cox-Time architecture
        in_features = X_train.shape[1]
        num_nodes_list = [params.get("nodes", 32)] * params.get("layers", 2)
        dropout = params.get("dropout", 0.2)

        net = MLPVanillaCoxTime(
            in_features=in_features,
            num_nodes=num_nodes_list,
            batch_norm=True,
            dropout=dropout
        )

        device = "cuda" if torch.cuda.is_available() else "cpu"

        # Canonical optimiser setup to prevent state-retention issues
        model = CoxTime(net, tt.optim.Adam, labtrans=labtrans, device=device)
        model.optimizer.set_lr(params.get("lr", 1e-3))

        callbacks = [tt.callbacks.EarlyStopping(patience=10)]

        # Train model
        model.fit(
            X_tr, y_tr,
            batch_size=params.get("batch_size", 256),
            epochs=200,
            callbacks=callbacks,
            verbose=False,
            val_data=(X_val, y_val)
        )

        # Predict
        model.compute_baseline_hazards()
        surv_df = model.predict_surv_df(X_test, batch_size=512)

        # Compute risk (using negative expected survival as a proxy for Cox-Time)
        expected_survival_time = surv_df.sum().values
        risk = -expected_survival_time

        # Compute survival probabilities at requested times
        surv = surv_df.reindex(times, method="ffill").bfill().values.T.astype("float32")

        return model, risk, surv

    except Exception as e:
        print(f"Cox-Time evaluation failed: {e}")
        return None, None, None

In [28]:
def optimise_coxtime(X_tr, T_tr, E_tr, X_val, T_val, E_val, n_trials=15):
    # Ensure float32 for PyTorch/PyCox compatibility
    X_tr = X_tr.astype("float32")
    T_tr = T_tr.astype("float32")
    E_tr = E_tr.astype("float32")
    X_val = X_val.astype("float32")
    T_val = T_val.astype("float32")
    E_val = E_val.astype("float32")

    # Transform the target variables for Cox-Time outside the loop for speed.
    labtrans = LabTransCoxTime()
    y_tr_cox = labtrans.fit_transform(T_tr, E_tr)
    y_val_cox = labtrans.transform(T_val, E_val)

    y_tr = (y_tr_cox[0], y_tr_cox[1])
    y_val = (y_val_cox[0], y_val_cox[1])

    def objective(trial):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Parameters to optimise
        n_nodes = trial.suggest_int("nodes", 32, 256, step=32)
        n_layers = trial.suggest_int("layers", 1, 4)
        dropout = trial.suggest_float("dropout", 0.1, 0.5)
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [64, 128, 256, 512])

        # Build the model architecture using CoxTime MLP
        in_features = X_tr.shape[1]
        num_nodes_list = [n_nodes] * n_layers

        net = MLPVanillaCoxTime(
            in_features=in_features,
            num_nodes=num_nodes_list,
            batch_norm=True,
            dropout=dropout
        )

        device = "cuda" if torch.cuda.is_available() else "cpu"

        # Canonical optimiser setup
        model = CoxTime(net, tt.optim.Adam, labtrans=labtrans, device=device)
        model.optimizer.set_lr(lr)
        callbacks = [tt.callbacks.EarlyStopping(patience=10)]

        # Train the model
        model.fit(
            X_tr, y_tr,
            batch_size=batch_size, epochs=200,
            callbacks=callbacks, verbose=False,
            val_data=(X_val, y_val)
        )

        # Compute baseline hazards
        model.compute_baseline_hazards()

        # Predict the full survival dataframe for the validation set
        surv_df = model.predict_surv_df(X_val)

        # Compute c-index
        ev = EvalSurv(surv_df, T_val, E_val, censor_surv="km")
        c_index = ev.concordance_td()

        # Memory cleanup
        del model, net, ev, surv_df
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return c_index

    # Run study
    optuna.logging.set_verbosity(optuna.logging.WARNING) #
    study = optuna.create_study(direction="maximize")

    study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True)

    return study.best_params

In [29]:
n_trials = 20

In [30]:
params_rsf = optimise_rsf(X_tr, T_tr, E_tr, X_val, T_val, E_val, n_trials=n_trials)

  0%|          | 0/20 [00:00<?, ?it/s]

In [31]:
print(params_rsf)

{'n_estimators': 100, 'min_samples_split': 6, 'min_samples_leaf': 10, 'max_features': 'sqrt'}


In [32]:
params_deepsurv = optimise_deepsurv(X_tr, T_tr, E_tr, X_val, T_val, E_val, n_trials=n_trials)

  0%|          | 0/20 [00:00<?, ?it/s]

In [33]:
print(params_deepsurv)

{'nodes': 176, 'layers': 1, 'dropout': 0.41246465660636494, 'lr': 0.00011621339500738244, 'l2': 0.017116424541481084, 'batch_size': 256, 'activation': 'ReLU', 'batch_norm': True}


In [34]:
# %% capture
params_coxtime = optimise_coxtime(X_tr, T_tr, E_tr, X_val, T_val, E_val, n_trials=n_trials)

  0%|          | 0/20 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torchtuples/tupletree.py:597: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  return self.tuple_.apply(lambda x: x[index])
/usr/local/lib/python3.12/dist-packages/torchtuples/tupletree.py:597: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  return self.tuple_.apply(lambda x: x[index]

In [35]:
print(params_coxtime)

{'nodes': 32, 'layers': 2, 'dropout': 0.10749734467390246, 'lr': 0.007331931202829721, 'batch_size': 512}


In [36]:
# params_rsf = {'n_estimators': 40, 'min_samples_split': 11, 'min_samples_leaf': 5, 'max_features': 'sqrt'}
# params_deepsurv = {'nodes': 192, 'layers': 1, 'dropout': 0.10786701980318078, 'lr': 0.0002890784908386592, 'l2': 0.0001977233138078638, 'batch_size': 32, 'activation': 'ReLU', 'batch_norm': False}
# params_coxtime = {'nodes': 256, 'layers': 3, 'dropout': 0.276256885400501, 'lr': 0.0063642779872328595, 'batch_size': 512}

In [37]:
def evaluate_seer(X_train, T_train, E_train,
                  X_test, T_test, E_test, params_rsf,
                  params_ds, params_dh,
                  n_bootstraps=1000):

    # ensure numpy arrays for indexing
    T_test = np.array(T_test)
    E_test = np.array(E_test)

    # format targets for IBS (sksurv)
    y_train_sksurv = np.array([(bool(e), t) for e, t in zip(E_train, T_train)], dtype=[("e", bool), ("t", float)])
    y_test_sksurv = np.array([(bool(e), t) for e, t in zip(E_test, T_test)], dtype=[("e", bool), ("t", float)])

    # define the time grid for IBS integration (5th to 95th percentile of test events)
    mask = E_test == 1
    event_times = T_test[mask]
    times = np.linspace(np.percentile(event_times, 5), np.percentile(event_times, 95), 100)

    # train all models and get predictions
    print("Training Cox-Time...")
    gc.collect()
    coxtime_model, risk_dh, surv_dh = coxtime(X_train, T_train, E_train, X_test, params_dh, times=times)
    print("Training Random Survival Forest...")
    gc.collect()
    rsf_model,risk_rsf, surv_rsf = rsf(X_train, T_train, E_train, X_test, params_rsf, times=times)
    gc.collect()
    print("Training CPH...")
    risk_cph, surv_cph = cph(X_train, T_train, E_train, X_test, times=times)
    print("Training Cox Lasso...")
    risk_lasso, surv_lasso = cox_lasso(X_train, T_train, E_train, X_test, times=times)
    print("Training DeepSurv...")
    deepsurv_model, risk_ds, surv_ds = deepsurv(X_train, T_train, E_train, X_test, params_ds, times=times)

    # memory cleanup
    gc.collect()

    # group predictions into a dictionary
    models = {
        "Cox Proportional Hazards": (risk_cph, surv_cph),
        "Cox Lasso": (risk_lasso, surv_lasso),
        "DeepSurv": (risk_ds, surv_ds),
        "Cox-Time": (risk_dh, surv_dh),
        "Random Survival Forest": (risk_rsf, surv_rsf)
    }

    # store fitted models
    fitted_models = {
        "Random Survival Forest": rsf_model,
        "DeepSurv": deepsurv_model,
        "Cox-Time": coxtime_model
    }

    # initialise a dictionary to hold bootstrap results
    boot_results = {name: {"cidx": [], "ibs": []} for name in models.keys()}

    # start bootstrapping
    for i in tqdm(range(n_bootstraps), desc="Bootstrapping SEER Colorectal Data"):

        # resample the test set indices with replacement
        indices = resample(np.arange(len(X_test)), replace=True, random_state=i)

        T_boot = T_test[indices]
        E_boot = E_test[indices]
        y_test_boot = y_test_sksurv[indices]

        # skip if no events are present
        if np.sum(E_boot) == 0:
            continue

        # create time mask to prevent sksurv from extrapolating
        time_mask = (times >= T_boot.min()) & (times < T_boot.max())
        times_boot = times[time_mask]

        if len(times_boot) < 2:
            continue

        # evaluate all models
        for name, (risk, surv) in models.items():
            # Concordance Index
            c_index = concordance_index(T_boot, -risk[indices], E_boot)
            boot_results[name]["cidx"].append(c_index)

            # Integrated Brier Score
            ibs = integrated_brier_score(y_train_sksurv, y_test_boot, surv[indices][:, time_mask], times_boot)
            boot_results[name]["ibs"].append(ibs)

        # periodic memory cleanup
        if i % 100 == 0:
            gc.collect()

    # output mean and 95% CI for each metric and model
    def get_metrics(boot_list):
        if not boot_list:
            return "NaN", "NaN"
        mean_val = round(np.mean(boot_list), 3)
        lower_ci = round(np.percentile(boot_list, 2.5), 3)
        upper_ci = round(np.percentile(boot_list, 97.5), 3)
        return f"{mean_val:.3f}", f"({lower_ci:.3f}, {upper_ci:.3f})"

    # build table
    final_rows = []
    for name in models.keys():
        mean_c, ci_c = get_metrics(boot_results[name]["cidx"])
        mean_ibs, ci_ibs = get_metrics(boot_results[name]["ibs"])

        final_rows.append({
            "Model": name,
            "C-Index (Mean)": mean_c,
            "C-Index (95% CI)": ci_c,
            "IBS (Mean)": mean_ibs,
            "IBS (95% CI)": ci_ibs
        })

    # Return the metrics dataframe, the predictions, and the time grid
    return pd.DataFrame(final_rows), models, times, fitted_models

In [38]:
metrics_df, predictions, time_grid, fitted_models = evaluate_seer(
    X_train, T_train, E_train,
    X_test, T_test, E_test,
    params_rsf=params_rsf,
    params_ds=params_deepsurv,
    params_dh=params_coxtime,
    n_bootstraps=100
)

Training Cox-Time...


/usr/local/lib/python3.12/dist-packages/torchtuples/tupletree.py:597: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  return self.tuple_.apply(lambda x: x[index])


Training Random Survival Forest...
building tree 1 of 100
building tree 2 of 100
building tree 3 of 100
building tree 4 of 100
building tree 5 of 100
building tree 6 of 100
building tree 7 of 100
building tree 8 of 100
building tree 9 of 100
building tree 10 of 100
building tree 11 of 100
building tree 12 of 100
building tree 13 of 100
building tree 14 of 100
building tree 15 of 100
building tree 16 of 100
building tree 17 of 100
building tree 18 of 100
building tree 19 of 100
building tree 20 of 100
building tree 21 of 100
building tree 22 of 100
building tree 23 of 100
building tree 24 of 100
building tree 25 of 100
building tree 26 of 100
building tree 27 of 100
building tree 28 of 100
building tree 29 of 100
building tree 30 of 100
building tree 31 of 100
building tree 32 of 100
building tree 33 of 100
building tree 34 of 100
building tree 35 of 100
building tree 36 of 100
building tree 37 of 100
building tree 38 of 100
building tree 39 of 100
building tree 40 of 100


[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:  1.4min


building tree 41 of 100
building tree 42 of 100
building tree 43 of 100
building tree 44 of 100
building tree 45 of 100
building tree 46 of 100
building tree 47 of 100
building tree 48 of 100
building tree 49 of 100
building tree 50 of 100
building tree 51 of 100
building tree 52 of 100
building tree 53 of 100
building tree 54 of 100
building tree 55 of 100
building tree 56 of 100
building tree 57 of 100
building tree 58 of 100
building tree 59 of 100
building tree 60 of 100
building tree 61 of 100
building tree 62 of 100
building tree 63 of 100
building tree 64 of 100
building tree 65 of 100
building tree 66 of 100
building tree 67 of 100
building tree 68 of 100
building tree 69 of 100
building tree 70 of 100
building tree 71 of 100
building tree 72 of 100
building tree 73 of 100
building tree 74 of 100
building tree 75 of 100
building tree 76 of 100
building tree 77 of 100
building tree 78 of 100
building tree 79 of 100
building tree 80 of 100
building tree 81 of 100
building tree 82

[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:  3.4min finished
[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:   14.4s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:   35.6s finished
[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:    5.8s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:   14.8s finished


Training CPH...
Training Cox Lasso...
Training DeepSurv...


Bootstrapping SEER Colorectal Data:   0%|          | 0/100 [00:00<?, ?it/s]

In [39]:
display(metrics_df)

,Model,C-Index (Mean),C-Index (95% CI),IBS (Mean),IBS (95% CI)
0,Cox Proportional Hazards,0.755,"(0.753, 0.758)",0.158,"(0.158, 0.159)"
1,Cox Lasso,0.755,"(0.753, 0.757)",0.159,"(0.158, 0.159)"
2,DeepSurv,0.770,"(0.768, 0.772)",0.151,"(0.150, 0.152)"
3,Cox-Time,0.771,"(0.769, 0.772)",0.149,"(0.148, 0.150)"
4,Random Survival Forest,0.767,"(0.765, 0.769)",0.149,"(0.148, 0.150)"


In [40]:
print(predictions)

{'Cox Proportional Hazards': (array([2.75037149, 1.52545257, 0.24396677, ..., 1.67218477, 1.3561511 ,
       1.23169066]), array([[0.95385587, 0.90529977, 0.86853503, ..., 0.0491754 , 0.04765968,
        0.04622856],
       [0.97413785, 0.94631463, 0.92480317, ..., 0.18810375, 0.18486564,
        0.18176591],
       [0.99581819, 0.99121382, 0.98757535, ..., 0.76551524, 0.76339228,
        0.76133057],
       ...,
       [0.97168573, 0.94130514, 0.91787516, ..., 0.16017778, 0.15715769,
        0.15427142],
       [0.97697483, 0.95212777, 0.9328618 , ..., 0.22642691, 0.22295835,
        0.2196317 ],
       [0.97906567, 0.95642402, 0.93883077, ..., 0.25949522, 0.25588236,
        0.25241247]])), 'Cox Lasso': (array([ 0.99391122,  0.42941106, -1.39503401, ...,  0.5213513 ,
        0.30570822,  0.21133676]), array([[0.95363122, 0.91888422, 0.89042599, ..., 0.05121291, 0.05018141,
        0.04816838],
       [0.97336296, 0.95303414, 0.93613631, ..., 0.18454232, 0.18241942,
        0.17822151

In [41]:
print(time_grid)

[  0.          1.4848485   2.969697    4.4545455   5.939394    7.4242425
   8.909091   10.39394    11.878788   13.363636   14.848485   16.333334
  17.818182   19.30303    20.78788    22.272728   23.757576   25.242424
  26.727272   28.212122   29.69697    31.181818   32.666668   34.151516
  35.636364   37.121212   38.60606    40.090908   41.57576    43.060608
  44.545456   46.030304   47.515152   49.         50.484848   51.969696
  53.454544   54.939396   56.424244   57.909092   59.39394    60.878788
  62.363636   63.848484   65.333336   66.818184   68.30303    69.78788
  71.27273    72.757576   74.242424   75.72727    77.21212    78.69697
  80.181816   81.666664   83.15152    84.63637    86.121216   87.606064
  89.09091    90.57576    92.06061    93.545456   95.030304   96.51515
  98.         99.48485   100.969696  102.454544  103.93939   105.42424
 106.90909   108.39394   109.87879   111.36364   112.84849   114.333336
 115.818184  117.30303   118.78788   120.27273   121.757576  123.24

In [42]:
# Save the results
from google.colab import drive
import joblib

drive.mount("/content/drive")

# path for predictions
save_path = "/content/drive/MyDrive/seer_predictions.joblib"

# Save it directly to Drive
joblib.dump(predictions, save_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


['/content/drive/MyDrive/seer_predictions.joblib']

In [43]:
# path for model results
save_path = "/content/drive/MyDrive/seer_model_metrics.joblib"

# Save it directly to Drive
joblib.dump(metrics_df, save_path)

['/content/drive/MyDrive/seer_model_metrics.joblib']

In [44]:
# path for times
save_path = "/content/drive/MyDrive/seer_time_grid.joblib"

# Save it directly to Drive
joblib.dump(time_grid, save_path)

['/content/drive/MyDrive/seer_time_grid.joblib']

In [45]:
# Save the Random Survival Forest using joblib
rsf_path = "/content/drive/MyDrive/seer_rsf_model.joblib"
joblib.dump(fitted_models["Random Survival Forest"], rsf_path)
print("RSF saved successfully.")

# Save Deepsurv using PyCox's built-in method
ds_path = "/content/drive/MyDrive/seer_deepsurv_model.pt"
fitted_models["DeepSurv"].save_net(ds_path)
print("DeepSurv saved successfully.")

# save Cox-Time model
ct_path = "/content/drive/MyDrive/seer_coxtime_model.pt"
fitted_models["Cox-Time"].save_net(ct_path)
print("Cox-Time saved successfully.")

RSF saved successfully.
DeepSurv saved successfully.
Cox-Time saved successfully.
